# Step 7-3a: 감성분석 모델 비교 (1000건 샘플)

3개 한국어 감성 모델을 별점과 비교해서 **가장 일치율 높은 모델 선정**.

## 비교 모델
| 모델 | 출력 | 도메인 |
|------|------|--------|
| **KoELECTRA-NSMC** | 2-class (긍/부) | 영화리뷰 |
| **KOTE** | 44 감정 multi-label | 한국어 댓글 |
| **KcELECTRA-multi** | 11 감정 multi-class | 한국어 리뷰 |

## 평가 방식
1. 별점별 stratified 1000건 (각 별점 200건)
2. 별점 → 3-class GT (5,4=긍 / 3=중 / 1,2=부)
3. 각 모델 → 3-class 라벨 매핑
4. **Accuracy + macro-F1 + class별 F1** 비교
5. 최고 모델 선정 → 7-3b로

## 입력 / 출력
| 파일 | 내용 |
|---|---|
| `embedding_input.parquet` | 평점·리뷰내용_clean 보유 |
| → `step7_3a_model_comparison.csv` | 모델별 평가 지표 |
| → `step7_3a_predictions_sample.csv` | 1000건 예측 결과 (검증용) |


## 0. 환경 셋업

In [1]:
!pip install -q transformers accelerate

In [2]:
from google.colab import drive

import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from tqdm.auto import tqdm

In [3]:
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/sparta/tp4/review_analysis/data/'
INPUT_PATH = OUTPUT_DIR + 'embedding_input.parquet'

print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'INPUT_PATH: {INPUT_PATH}')

Mounted at /content/drive
OUTPUT_DIR: /content/drive/MyDrive/sparta/tp4/review_analysis/data/
INPUT_PATH: /content/drive/MyDrive/sparta/tp4/review_analysis/data/embedding_input.parquet


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB


## 7-3a-1. 1000건 stratified 샘플링

별점이 5점에 편중되어 있으므로 **각 별점 200건씩 균등 추출** — 부정 클래스 검증력 확보.

In [5]:
df = pd.read_parquet(INPUT_PATH)
df['리뷰내용_clean'] = df['리뷰내용_clean'].fillna('').astype(str)

# 너무 짧은 리뷰는 평가에서 제외 (모델 신뢰도 낮음)
df = df[df['리뷰내용_clean'].str.len() >= 5].reset_index(drop=True)

print(f'전체 리뷰: {len(df):,}')
print(f'\n[평점 분포]')
print(df['평점'].value_counts().sort_index())

전체 리뷰: 598,301

[평점 분포]
평점
1.0      1587
2.0      2013
3.0     15953
4.0     85350
5.0    491770
Name: count, dtype: int64


In [6]:
# # 별점별 stratified 샘플링
# SAMPLE_PER_RATING = 200
# SEED = 42

# samples = []
# for rating in [1, 2, 3, 4, 5]:
#     sub = df[df['평점'] == rating]
#     n_take = min(len(sub), SAMPLE_PER_RATING)
#     samples.append(sub.sample(n_take, random_state=SEED))
#     print(f'  평점 {rating}: {n_take}건 추출 (전체 {len(sub):,})')

# df_sample = pd.concat(samples).reset_index(drop=True)
# print(f'\n샘플 합계: {len(df_sample)}건')

# # GT 매핑: 별점 → 3-class
# def rating_to_class(r):
#     if r >= 4: return 'positive'
#     if r == 3: return 'neutral'
#     return 'negative'

# df_sample['gt_label'] = df_sample['평점'].apply(rating_to_class)
# print(f'\n[GT 분포]')
# print(df_sample['gt_label'].value_counts())
# 전체 데이터 사용
SEED = 42

df_sample = df.copy()
print(f'전체 데이터: {len(df_sample)}건')

# GT 매핑: 별점 → 3-class
def rating_to_class(r):
    if r >= 4: return 'positive'
    if r == 3: return 'neutral'
    return 'negative'

df_sample['gt_label'] = df_sample['평점'].apply(rating_to_class)
print(f'\n[GT 분포]')
print(df_sample['gt_label'].value_counts())

전체 데이터: 598301건

[GT 분포]
gt_label
positive    577120
neutral      15953
negative      5228
Name: count, dtype: int64


## 7-3a-2. 추론 함수 (배치 처리)

GPU 배치로 1000건 빠르게 추론.

In [7]:
@torch.no_grad()
def predict_logits(texts, model, tokenizer, batch_size=32, max_length=128):
    """배치로 logits 추출 (모델별 후처리는 별도)."""
    model.eval()
    all_logits = []
    for i in tqdm(range(0, len(texts), batch_size), desc='추론'):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=max_length, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
        all_logits.append(outputs.logits.cpu())
    return torch.cat(all_logits, dim=0)


def evaluate(gt, pred, model_name):
    """3-class 평가 결과 출력 + 지표 반환."""
    print(f'\n{"="*60}')
    print(f'{model_name}')
    print("="*60)

    acc = accuracy_score(gt, pred)
    f1_macro = f1_score(gt, pred, labels=['negative', 'neutral', 'positive'],
                        average='macro', zero_division=0)
    f1_per = f1_score(gt, pred, labels=['negative', 'neutral', 'positive'],
                      average=None, zero_division=0)

    print(f'Accuracy : {acc:.4f}')
    print(f'F1 macro : {f1_macro:.4f}')
    print(f'F1 (neg) : {f1_per[0]:.4f}')
    print(f'F1 (neu) : {f1_per[1]:.4f}')
    print(f'F1 (pos) : {f1_per[2]:.4f}')

    print(f'\n[Confusion Matrix]')
    cm = confusion_matrix(gt, pred, labels=['negative', 'neutral', 'positive'])
    print(pd.DataFrame(cm,
                       index=['GT_neg', 'GT_neu', 'GT_pos'],
                       columns=['P_neg', 'P_neu', 'P_pos']))

    return {
        'model': model_name,
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_neg': f1_per[0],
        'f1_neu': f1_per[1],
        'f1_pos': f1_per[2],
    }

## 7-3a-3. 모델 1: KoELECTRA-NSMC (2-class)

가장 검증된 한국어 감성 모델. 출력은 0=부정/1=긍정 두 클래스.
**중립 도출**: 긍정 확률이 0.4~0.6 사이면 중립으로 분류.

In [8]:
# 모델 로드
M1_ID = 'monologg/koelectra-base-finetuned-nsmc'
print(f'로드: {M1_ID}')
tok1 = AutoTokenizer.from_pretrained(M1_ID)
m1 = AutoModelForSequenceClassification.from_pretrained(M1_ID).to(device)
print(f'  라벨: {m1.config.id2label}')

로드: monologg/koelectra-base-finetuned-nsmc


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/102 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  라벨: {0: 'negative', 1: 'positive'}


In [9]:
# 추론
texts = df_sample['리뷰내용_clean'].tolist()
t0 = time.time()
logits1 = predict_logits(texts, m1, tok1)
print(f'\n추론 완료: {time.time()-t0:.1f}초')

# 매핑: 긍정 확률 → 3-class
# - P(pos) > 0.6 → positive
# - P(pos) < 0.4 → negative
# - 그 외 → neutral
probs1 = F.softmax(logits1, dim=-1).numpy()
p_pos1 = probs1[:, 1]  # 긍정 확률 (라벨 1=긍정)

NEU_LOW, NEU_HIGH = 0.4, 0.6
pred1 = []
for p in p_pos1:
    if p > NEU_HIGH:   pred1.append('positive')
    elif p < NEU_LOW:  pred1.append('negative')
    else:              pred1.append('neutral')

df_sample['pred_nsmc'] = pred1
df_sample['prob_pos_nsmc'] = p_pos1

# 평가
res1 = evaluate(df_sample['gt_label'], pred1, 'KoELECTRA-NSMC')

# GPU 메모리 정리
del m1, tok1
torch.cuda.empty_cache()

추론:   0%|          | 0/18697 [00:00<?, ?it/s]


추론 완료: 543.2초

KoELECTRA-NSMC
Accuracy : 0.8919
F1 macro : 0.3685
F1 (neg) : 0.1089
F1 (neu) : 0.0483
F1 (pos) : 0.9484

[Confusion Matrix]
        P_neg  P_neu   P_pos
GT_neg   2975    123    2130
GT_neu   6899    583    8471
GT_pos  39552   7500  530068


## 7-3a-4. 모델 2: KOTE (44 감정 multi-label)

44개 한국어 감정. **다중 라벨**(sigmoid)이라 한 텍스트에 여러 감정이 동시에 활성화될 수 있음.

### 매핑 전략
- 활성화된 감정(threshold 이상) 중 **긍정 감정 합 vs 부정 감정 합** 비교
- 둘 다 약하면 → neutral

In [10]:
# KOTE 44 감정 → 3-class 매핑
# (KOTE 데이터셋 라벨 기준)
KOTE_POSITIVE = {
    '환영/호의', '감동/감탄', '고마움', '존경', '기대감', '뿌듯함',
    '편안/쾌적', '신기함/관심', '아껴주는', '즐거움/신남', '깨달음',
    '흐뭇함(귀여움/예쁨)', '행복', '기쁨', '안심/신뢰',
}
KOTE_NEGATIVE = {
    '불평/불만', '지긋지긋', '슬픔', '화남/분노', '안타까움/실망',
    '의심/불신', '부끄러움', '공포/무서움', '절망', '한심함',
    '역겨움/징그러움', '짜증', '어이없음', '패배/자기혐오',
    '귀찮음', '힘듦/지침', '죄책감', '증오/혐오', '당황/난처',
    '경악', '부담/안 내킴', '서러움', '재미없음', '불쌍함/연민',
    '불안/걱정', '비장함',
}
KOTE_NEUTRAL = {
    '없음', '놀람', '우쭐댐/무시함',
}

# 모델 로드
M2_ID = 'searle-j/kote_for_easygoing_people'
print(f'로드: {M2_ID}')
tok2 = AutoTokenizer.from_pretrained(M2_ID)
m2 = AutoModelForSequenceClassification.from_pretrained(M2_ID).to(device)

# id2label 확인 (KOTE 라벨이 우리 매핑과 일치하는지)
print(f'\n[모델 라벨 (총 {len(m2.config.id2label)}개)]')
labels_kote = list(m2.config.id2label.values())
print(labels_kote[:10], '...')

# 매핑 누락 체크
missing = set(labels_kote) - KOTE_POSITIVE - KOTE_NEGATIVE - KOTE_NEUTRAL
if missing:
    print(f'\n매핑 누락 라벨 ({len(missing)}개): {missing}')
    print('   → 이 라벨들은 자동으로 neutral로 처리됨')

로드: searle-j/kote_for_easygoing_people


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/545 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: searle-j/kote_for_easygoing_people
Key                             | Status     |  | 
--------------------------------+------------+--+-
electra.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[모델 라벨 (총 44개)]
['불평/불만', '환영/호의', '감동/감탄', '지긋지긋', '고마움', '슬픔', '화남/분노', '존경', '기대감', '우쭐댐/무시함'] ...

매핑 누락 라벨 (1개): {'부담/안_내킴'}
   → 이 라벨들은 자동으로 neutral로 처리됨


In [11]:
# 추론
t0 = time.time()
logits2 = predict_logits(df_sample['리뷰내용_clean'].tolist(), m2, tok2)
print(f'\n추론 완료: {time.time()-t0:.1f}초')

# 매핑: 다중 라벨 sigmoid
probs2 = torch.sigmoid(logits2).numpy()
id2label2 = m2.config.id2label

# 각 샘플별 긍정/부정 점수 합산
ACTIVATION_THRESHOLD = 0.3  # 이 이상이면 활성화로 간주

pred2 = []
pos_scores2 = []
neg_scores2 = []
for i in range(len(df_sample)):
    pos_sum = 0.0
    neg_sum = 0.0
    for j, score in enumerate(probs2[i]):
        if score < ACTIVATION_THRESHOLD:
            continue
        label = id2label2[j]
        if label in KOTE_POSITIVE:
            pos_sum += score
        elif label in KOTE_NEGATIVE:
            neg_sum += score
    pos_scores2.append(pos_sum)
    neg_scores2.append(neg_sum)

    # 결정: 차이가 크면 그쪽, 둘 다 약하면 neutral
    diff = pos_sum - neg_sum
    if pos_sum < 0.2 and neg_sum < 0.2:
        pred2.append('neutral')
    elif diff > 0.15:
        pred2.append('positive')
    elif diff < -0.15:
        pred2.append('negative')
    else:
        pred2.append('neutral')

df_sample['pred_kote'] = pred2
df_sample['kote_pos_score'] = pos_scores2
df_sample['kote_neg_score'] = neg_scores2

res2 = evaluate(df_sample['gt_label'], pred2, 'KOTE')

del m2, tok2
torch.cuda.empty_cache()

추론:   0%|          | 0/18697 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]


추론 완료: 519.0초

KOTE
Accuracy : 0.9132
F1 macro : 0.3727
F1 (neg) : 0.1327
F1 (neu) : 0.0245
F1 (pos) : 0.9609

[Confusion Matrix]
        P_neg  P_neu   P_pos
GT_neg   3175     30    2023
GT_neu   7742    228    7983
GT_pos  31716   2432  542972


## 7-3a-5. 모델 3: nlp04/KcELECTRA-multi (다중 클래스)

한국어 리뷰 감정 분류 모델. softmax(단일 라벨).

> 모델 로드 후 `id2label`을 보고 우리 매핑(`KCELECTRA_*_LABELS`)이 맞는지 확인.
> 만약 라벨명이 다르면 매핑 사전을 수정해야 함.

In [12]:
# 예상 라벨 (모델 카드 기준 — 코랩 환경에서 실제 확인 필요)
# 7-3a-5 셀 매핑 사전 교체 (KcELECTRA 실제 라벨 기준)
KCELECTRA_POSITIVE = {
    '기쁨(행복한)', '고마운', '설레는(기대하는)',
    '사랑하는', '즐거운(신나는)',
}
KCELECTRA_NEGATIVE = {
    '슬픔(우울한)', '힘듦(지침)', '짜증남', '걱정스러운(불안한)',
}
KCELECTRA_NEUTRAL = {
    '일상적인', '생각이 많은',
}

# 모델 로드
M3_ID = 'nlp04/korean_sentiment_analysis_kcelectra'
print(f'로드: {M3_ID}')
tok3 = AutoTokenizer.from_pretrained(M3_ID)
m3 = AutoModelForSequenceClassification.from_pretrained(M3_ID).to(device)

print(f'\n[모델 라벨 (총 {len(m3.config.id2label)}개)]')
for idx, lab in m3.config.id2label.items():
    print(f'  {idx}: {lab}')

# 매핑 검증
labels_m3 = set(m3.config.id2label.values())
mapped = KCELECTRA_POSITIVE | KCELECTRA_NEGATIVE | KCELECTRA_NEUTRAL
missing = labels_m3 - mapped
if missing:
    print(f'\n매핑되지 않은 라벨: {missing}')
    print('   → 자동으로 neutral 처리. 위 라벨을 보고 KCELECTRA_* 사전 수정 권장')

로드: nlp04/korean_sentiment_analysis_kcelectra


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/511M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: nlp04/korean_sentiment_analysis_kcelectra
Key                             | Status     |  | 
--------------------------------+------------+--+-
electra.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[모델 라벨 (총 11개)]
  0: 기쁨(행복한)
  1: 고마운
  2: 설레는(기대하는)
  3: 사랑하는
  4: 즐거운(신나는)
  5: 일상적인
  6: 생각이 많은
  7: 슬픔(우울한)
  8: 힘듦(지침)
  9: 짜증남
  10: 걱정스러운(불안한)


In [13]:
# 추론 (단일 라벨 softmax)
t0 = time.time()
logits3 = predict_logits(df_sample['리뷰내용_clean'].tolist(), m3, tok3)
print(f'\n추론 완료: {time.time()-t0:.1f}초')

probs3 = F.softmax(logits3, dim=-1).numpy()
id2label3 = m3.config.id2label

# 매핑
pred3 = []
for i in range(len(df_sample)):
    top_idx = probs3[i].argmax()
    top_label = id2label3[top_idx]
    if top_label in KCELECTRA_POSITIVE:
        pred3.append('positive')
    elif top_label in KCELECTRA_NEGATIVE:
        pred3.append('negative')
    else:
        pred3.append('neutral')

df_sample['pred_kcelectra'] = pred3
df_sample['kcelectra_top_label'] = [id2label3[probs3[i].argmax()]
                                     for i in range(len(df_sample))]

res3 = evaluate(df_sample['gt_label'], pred3, 'KcELECTRA-multi')

del m3, tok3
torch.cuda.empty_cache()

추론:   0%|          | 0/18697 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/511M [00:00<?, ?B/s]


추론 완료: 509.8초

KcELECTRA-multi
Accuracy : 0.8098
F1 macro : 0.3495
F1 (neg) : 0.0866
F1 (neu) : 0.0599
F1 (pos) : 0.9021

[Confusion Matrix]
        P_neg  P_neu   P_pos
GT_neg   3318    418    1492
GT_neu   9323   1705    4925
GT_pos  58731  38881  479508


## 7-3a-6. 종합 비교

세 모델의 지표를 한 표로 정리.

### 권장 선정 기준
1. **F1 macro 우선** — 클래스 불균형(별점 5 편중)을 고려
2. **F1 (neg) 가중 고려** — 부정 케이스 잡기가 ABSA의 가치
3. Accuracy는 보조 지표 (편향 가능성)

In [14]:
results_df = pd.DataFrame([res1, res2, res3])
results_df = results_df.sort_values('f1_macro', ascending=False).reset_index(drop=True)
results_df['rank_f1_macro'] = results_df['f1_macro'].rank(ascending=False).astype(int)
results_df['rank_f1_neg']   = results_df['f1_neg'].rank(ascending=False).astype(int)

print('[모델 비교 종합]')
print(results_df.to_string(index=False))

# 추천
best_model = results_df.iloc[0]['model']
best_f1 = results_df.iloc[0]['f1_macro']
print(f'\n추천 모델: {best_model} (F1 macro: {best_f1:.4f})')

# 저장
comp_path = OUTPUT_DIR + 'step7_3a_model_comparison.csv'
results_df.to_csv(comp_path, index=False, encoding='utf-8-sig')
print(f'\n저장: {comp_path}')

[모델 비교 종합]
          model  accuracy  f1_macro   f1_neg   f1_neu   f1_pos  rank_f1_macro  rank_f1_neg
           KOTE  0.913211  0.372688 0.132676 0.024460 0.960929              1            1
 KoELECTRA-NSMC  0.891902  0.368518 0.108867 0.048264 0.948422              2            2
KcELECTRA-multi  0.809845  0.349547 0.086632 0.059870 0.902141              3            3

추천 모델: KOTE (F1 macro: 0.3727)

저장: /content/drive/MyDrive/sparta/tp4/review_analysis/data/step7_3a_model_comparison.csv


In [15]:
# 모델 간 의견 일치도 (sanity check)
agreement_all = (
    (df_sample['pred_nsmc'] == df_sample['pred_kote']) &
    (df_sample['pred_nsmc'] == df_sample['pred_kcelectra'])
).sum()

agree_pairs = pd.DataFrame({
    'NSMC vs KOTE':       (df_sample['pred_nsmc'] == df_sample['pred_kote']).sum(),
    'NSMC vs KcELECTRA':  (df_sample['pred_nsmc'] == df_sample['pred_kcelectra']).sum(),
    'KOTE vs KcELECTRA':  (df_sample['pred_kote'] == df_sample['pred_kcelectra']).sum(),
}, index=['일치 건수'])

print('[모델 간 일치도]')
print(agree_pairs.T)
print(f'\n3개 모두 일치: {agreement_all}/{len(df_sample)} ({agreement_all/len(df_sample)*100:.1f}%)')

[모델 간 일치도]
                    일치 건수
NSMC vs KOTE       549447
NSMC vs KcELECTRA  505774
KOTE vs KcELECTRA  523508

3개 모두 일치: 496674/598301 (83.0%)


In [16]:
# 모델별 불일치 케이스 샘플 — 어떤 케이스에서 갈리는지
print('[3개 모델 모두 GT(별점)와 다른 케이스 샘플]')
all_wrong = df_sample[
    (df_sample['pred_nsmc']      != df_sample['gt_label']) &
    (df_sample['pred_kote']      != df_sample['gt_label']) &
    (df_sample['pred_kcelectra'] != df_sample['gt_label'])
]
print(f'총 {len(all_wrong)}건')

cols_show = ['평점', 'gt_label', 'pred_nsmc', 'pred_kote', 'pred_kcelectra', '리뷰내용_clean']
cols_show = [c for c in cols_show if c in df_sample.columns]
sample = all_wrong[cols_show].sample(min(15, len(all_wrong)), random_state=42)

pd.set_option('display.max_colwidth', 100)
display(sample)

[3개 모델 모두 GT(별점)와 다른 케이스 샘플]
총 34758건


,평점,gt_label,pred_nsmc,pred_kote,pred_kcelectra,리뷰내용_clean
135938,NaN,negative,positive,positive,positive,xs인데도 155기준 어어엉엄청 커요! 발목 스트랩조절 가능이지만 굽낮은 운동화는 아슬아슬해요 근데 커서 다리 길어보임. 글고 넘 예뻐요ㅜㅜ
52921,3.0,neutral,positive,positive,negative,177 / 60 / 마른체형 / M 주문 원래 박시하게 입는 편인데 M 입었을 때 기장 빼고는 박시한 느낌나요 기장이 조금만 길었으면 완전 굿인데 ㅠ 그래도 예뻐요
294183,3.0,neutral,positive,positive,negative,입을만 해요 단점이 있으면 실밥이 쫌 있네요 가격대비 좋아요
190932,5.0,positive,negative,negative,negative,진짜 오버핏이네요 그래도 색감은 좋은거 같아서 그냥 둡니다..
557797,4.0,positive,negative,negative,negative,키가 있어서 평소입는 사이즈로 구매했는데 길이가 많이 긴편이네요...
383063,4.0,positive,negative,negative,negative,길고 너무 두꺼워서 여름에 입기에는 무리가 있음
181578,3.0,neutral,negative,negative,negative,핏은 예쁘고 좋은데 안감이 좀 거칠고 엇에 먼지가 좀 많이 붙네요
110112,5.0,positive,negative,negative,negative,옷은 이쁜데 블프 겹쳐서 배송이 좀 느렸어요 다른건 다 즇아요
511876,4.0,positive,neutral,negative,negative,재질은 부드럽고 좋아요. 크기도 적당하네요. 다만 목 부분이 잘 늘어날 것 같아요. 특히 검정색은 배송받아서 봤을 때부터 이미 살짝 늘어난 듯 쭈글한 상태예요. 행사할때 사...
215601,3.0,neutral,positive,positive,positive,무난한 색상이라 편하게 매치해서 입기 좋음 요즘 입기좋은 두께감~


## 7-3a-7. 예측 결과 저장

전체 1000건 예측 결과를 저장 — 7-3b에서 모델 선정 후 같은 매핑/threshold를 재사용.

In [17]:
# 예측 결과 저장 (검증용)
keep_cols = ['리뷰번호', '평점', '리뷰내용_clean', 'gt_label',
             'pred_nsmc', 'pred_kote', 'pred_kcelectra',
             'prob_pos_nsmc', 'kote_pos_score', 'kote_neg_score',
             'kcelectra_top_label']
keep_cols = [c for c in keep_cols if c in df_sample.columns]

pred_path = OUTPUT_DIR + 'step7_3a_predictions_sample.csv'
df_sample[keep_cols].to_csv(pred_path, index=False, encoding='utf-8-sig')
print(f'저장: {pred_path}')
print(f'  총 {len(df_sample):,}건 × {len(keep_cols)}컬럼')

저장: /content/drive/MyDrive/sparta/tp4/review_analysis/data/step7_3a_predictions_sample.csv
  총 598,301건 × 11컬럼
